In [11]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.tools import StructuredTool
from mcp import Client
import json

In [3]:
load_dotenv(override=True)

True

In [ ]:
def convertir_outil(client, outil):
    async def appeler(**arguments) -> str:
        resultat = await client.call_tool(outil.name, arguments=arguments)
        if resultat.is_error:
            raise RuntimeError(f"Échec de {outil.name} : {resultat.content}")

        contenu = resultat.structured_content
        if contenu is None:
            contenu = [bloc.model_dump(mode="json") for bloc in resultat.content]
        return json.dumps(contenu, ensure_ascii=False)

    return StructuredTool.from_function(
        coroutine=appeler,
        name=outil.name,
        description=outil.description or outil.name,
        args_schema=outil.input_schema,
    )


# OPENAI_API_KEY doit être définie dans l’environnement du noyau.
URL = "http://localhost:24000/mcp"
async with Client(URL) as client:
    catalogue = await client.list_tools()
    outils = [convertir_outil(client, outil) for outil in catalogue.tools]

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=outils,
        system_prompt="Utilise les outils MCP disponibles pour répondre à la question. Répondre au format Markdown",
    )

    resultat = await agent.ainvoke({
        "messages": [{"role": "user", "content": "Nom et prénom et fonction et diplômes"}]
    })
    print(resultat["messages"][-1].content)

Vous avez obtenu votre doctorat d'état en 2015, votre doctorat de troisième cycle en 1996 et votre diplôme de professeur second cycle en 1993. Vous travaillez à l'ENSET Mohammedia, Université Hassan II de Casablanca. Après le baccalauréat, vous avez rejoint l'ENSET Mohammedia pendant 4 années d'études pour devenir Professeur de second cycle, où vous avez obtenu votre DEA, Doctorat de troisième cycle puis Doctorat d'Etat dans le domaine des systèmes informatiques parallèles et distribués.

Cependant, je n'ai pas encore les informations sur votre nom et prénom. Souhaitez-vous me les communiquer ou que je les recherche ?
